# Laboratorio: RAG en GCP Agent Platform — Enunciados de Pensamiento Computacional

Este notebook construye un pipeline de **RAG (Retrieval-Augmented Generation)** sobre
Vertex AI **RAG Engine** (dentro de la consola renombrada a *Agent Platform*), usando como
base documental los **enunciados de los ejercicios del curso de Pensamiento Computacional**.

El flujo es el mismo que verías en la consola bajo *Agent Platform → RAG Engine*, pero
ejecutado por código para que sea reproducible: crear un **corpus**, **importar** los
documentos desde Cloud Storage, y luego **consultarlo** de dos formas — recuperación
directa (qué fragmentos trae) y generación anclada con Gemini (una respuesta redactada
citando el contenido real de los enunciados, no lo que el modelo ya sabía de antes).

> **SDK usado:** `google-cloud-agentplatform` (paquete actual del RAG Engine, agosto 2026 en
> adelante) + `google-genai` para la generación con Gemini. Es el mismo API que documenta
> el quickstart oficial de Google Cloud a la fecha de este notebook (septiembre 2026); si tu
> versión instalada difiere, revisa `docs.cloud.google.com/vertex-ai/generative-ai/docs/rag-engine`
> antes de correr las celdas — este ecosistema cambia de nombre de paquete con frecuencia.

## Paso 0 — Prerrequisitos (una sola vez, desde tu terminal)

```bash
# 1. Habilitar las APIs necesarias
gcloud services enable aiplatform.googleapis.com storage.googleapis.com

# 2. Autenticarte (si no lo has hecho ya en este proyecto)
gcloud auth login
gcloud auth application-default login
gcloud config set project TU_PROJECT_ID

# 3. Crear un bucket REGIONAL para los enunciados.
#    IMPORTANTE: el modo "Spanner" de RAG Engine (el que usa por defecto el quickstart
#    oficial) está restringido a proyectos en lista blanca en us-central1, us-east1 y
#    us-east4 por capacidad — en un proyecto nuevo como este da el error
#    "INVALID_ARGUMENT: ... is restricted to only allowlisted projects". Por eso usamos
#    us-west1 en vez de us-east4: queda fuera de esa restricción y no requiere ningún paso
#    adicional de configuración. Es independiente del bucket que usaste para el
#    fine-tuning en us-central1; no hay problema en tener buckets en distintas regiones
#    dentro del mismo proyecto.
gcloud storage buckets create gs://si7016_ragwest --location=us-west1

# 4. Subir los enunciados (PDF, DOCX, TXT, etc.) a ese bucket
gcloud storage cp -r ./docs/* gs://si7016_ragwest/pensamiento-computacional/
```

Con eso, tus documentos ya están en Cloud Storage y listos para importarse al corpus.

## Paso 1 — Instalar dependencias

In [ ]:
# pandas es requerido internamente por agentplatform (utilidades de GCS) pero no se
# instala solo como dependencia — sin esto falla con ModuleNotFoundError: No module named 'pandas'
!pip install --upgrade google-cloud-agentplatform google-genai pandas

## Paso 1b — Autenticación

- **Google Colab**: corre la celda de abajo; te va a pedir que inicies sesión con la
  cuenta que tiene acceso al proyecto de GCP.
- **Vertex AI Workbench**: no necesitas hacer nada — la instancia ya está autenticada con
  la cuenta de servicio del proyecto; puedes saltarte esta celda (no falla si la corres,
  simplemente no hace nada fuera de Colab).
- **Jupyter local**: asegúrate de haber corrido `gcloud auth application-default login`
  en tu terminal (Paso 0) *antes* de abrir este notebook; tampoco necesitas esta celda.

In [ ]:
try:
    import google.colab  # solo existe dentro de Colab
    from google.colab import auth
    auth.authenticate_user()
    print("Autenticado en Colab.")
except ImportError:
    print("No estás en Colab — se usarán las credenciales de gcloud/Workbench ya configuradas.")

## Paso 2 — Configuración

Ajusta las cuatro variables de abajo a tu proyecto, bucket y modelo. `MODEL_ID` es el
modelo de Gemini que redactará las respuestas ancladas en tus documentos; usa el que
tengas habilitado en tu proyecto (revisa el listado de modelos disponibles en la consola
de Agent Platform si `gemini-3.5-flash` no está disponible para ti todavía).

In [ ]:
import agentplatform
from agentplatform import types
from google import genai
from google.genai import types as genai_types

PROJECT_ID = "myproyectsi4002-262"          # TODO: tu project id
LOCATION = "us-west1"                        # región del RAG Engine (ver nota de la restricción de Spanner mode en el Paso 0)
GCS_PATH = "gs://si7016_ragwest/pensamiento-computacional/*"  # TODO: tu bucket real
CORPUS_DISPLAY_NAME = "enunciados-pensamiento-computacional"
MODEL_ID = "gemini-3.5-flash"
BUCKET_NAME = "si7016_ragwest" 
GCS_PREFIX = "pensamiento-computacional/"
GENAI_LOCATION = "global" 

client = agentplatform.Client(project=PROJECT_ID, location=LOCATION)
print("Cliente inicializado para", PROJECT_ID, "en", LOCATION)

Cliente inicializado para myproyectsi4002-262 en us-west1


## Paso 3 — Crear el corpus RAG

Un *corpus* es el índice vectorial donde vivirán los enunciados. Se crea una sola vez;
si vuelves a correr esta celda en una sesión futura, primero revisa en la consola
(*Agent Platform → RAG Engine*) si el corpus ya existe, para no duplicarlo.

`text-embedding-005` es el modelo de embeddings por defecto recomendado por Google para
RAG Engine.

In [2]:
embedding_model_config = types.RagEmbeddingModelConfig(
    vertex_prediction_endpoint=types.RagEmbeddingModelConfigVertexPredictionEndpoint(
        endpoint="publishers/google/models/text-embedding-005"
    ),
)

rag_corpus = client.rag.create_corpus(
    rag_corpus=types.RagCorpus(
        display_name=CORPUS_DISPLAY_NAME,
        rag_vector_db_config=types.RagVectorDbConfig(
            rag_embedding_model_config=embedding_model_config
        ),
    )
)
print("Corpus creado:", rag_corpus.name)

/var/folders/lg/hjr009s55m5_9q1dw9pwbp5r0000gn/T/ipykernel_5794/3772981002.py:7: ExperimentalWarning: The Vertex SDK GenAI rag module is experimental, and may change in future versions.
  rag_corpus = client.rag.create_corpus(


Corpus creado: projects/27569205695/locations/us-west1/ragCorpora/1152921504606846976


## Paso 4 — Importar los enunciados desde Cloud Storage

Trocea (*chunking*) cada documento en fragmentos de ~512 tokens con solape de 100, y los
vectoriza dentro del corpus. **La importación es asíncrona**: para pocos documentos suele
tardar 1-3 minutos, pero con más archivos o archivos grandes puede tomar más — antes de
pasar al Paso 5, confirma en la consola (*Agent Platform → RAG Engine → tu corpus → Files*)
que los archivos ya aparecen con estado completado.

In [9]:
from google.cloud import storage
storage_client = storage.Client(project=PROJECT_ID)
bucket = storage_client.bucket(BUCKET_NAME)

# delimiter="/" hace que la API devuelva las subcarpetas de primer nivel en .prefixes,
# en vez de listar recursivamente todos los archivos.
iterator = bucket.list_blobs(prefix=GCS_PREFIX, delimiter="/")
list(iterator)  # hay que consumir el iterador para que se llene .prefixes
subfolders = sorted(iterator.prefixes)

if subfolders:
    GCS_URIS = [f"gs://{BUCKET_NAME}/{prefix}*" for prefix in subfolders]
else:
    # No hay subcarpetas: los archivos están directo bajo GCS_PREFIX.
    GCS_URIS = [f"gs://{BUCKET_NAME}/{GCS_PREFIX}*"]

print(f"{len(GCS_URIS)} ruta(s) a importar:")
for uri in GCS_URIS:
    print(" -", uri)

8 ruta(s) a importar:
 - gs://si7016_ragwest/pensamiento-computacional/w1/*
 - gs://si7016_ragwest/pensamiento-computacional/w2/*
 - gs://si7016_ragwest/pensamiento-computacional/w3/*
 - gs://si7016_ragwest/pensamiento-computacional/w4/*
 - gs://si7016_ragwest/pensamiento-computacional/w5/*
 - gs://si7016_ragwest/pensamiento-computacional/w6/*
 - gs://si7016_ragwest/pensamiento-computacional/w7/*
 - gs://si7016_ragwest/pensamiento-computacional/w8/*


In [ ]:
PROJECT_NUMBER=27569205695   # el mismo que aparece en tus errores (projects/27569205695/...)

gcloud storage buckets add-iam-policy-binding gs://si7016_ragwest \
    --member="serviceAccount:service-${PROJECT_NUMBER}@gcp-sa-vertex-rag.iam.gserviceaccount.com" \
    --role="roles/storage.objectViewer"

In [19]:
import_response = client.rag.import_files(
    name=rag_corpus.name,
    import_config=types.ImportRagFilesConfig(
        gcs_source=genai_types.GcsSource(uris=[GCS_PATH]),
        rag_file_transformation_config=types.RagFileTransformationConfig(
            rag_file_chunking_config=types.RagFileChunkingConfig(
                chunk_size=512,
                chunk_overlap=100,
            )
        ),
        max_embedding_requests_per_min=1000,
    ),
)
print(import_response)

RuntimeError: Operation projects/27569205695/locations/us-west1/ragCorpora/1152921504606846976/operations/4861908545349615616 failed to import files into RagCorpus: {'code': 5, 'message': 'gs://si7016_ragwest/pensamiento-computacional/* either does not exist or is empty.'}

In [12]:
import_response = client.rag.import_files(
    name=rag_corpus.name,
    import_config=types.ImportRagFilesConfig(
        gcs_source=genai_types.GcsSource(uris=GCS_URIS),
        rag_file_transformation_config=types.RagFileTransformationConfig(
            rag_file_chunking_config=types.RagFileChunkingConfig(
                chunk_size=512,
                chunk_overlap=100,
            )
        ),
        max_embedding_requests_per_min=1000,
    ),
)
print(import_response)

RuntimeError: Operation projects/27569205695/locations/us-west1/ragCorpora/1152921504606846976/operations/3397323872779894784 failed to import files into RagCorpus: {'code': 5, 'message': 'gs://si7016_ragwest/pensamiento-computacional/w1/* either does not exist or is empty.'}

para el demo, se cargaron a mano por la interfaz consola


## Paso 5 — Recuperación directa (sin generación)

Antes de pedirle a Gemini que redacte una respuesta, vale la pena ver **qué fragmentos**
trae el corpus para una pregunta — así verificas que la importación funcionó y que el
contenido recuperado sí corresponde a tus enunciados.

In [13]:
PREGUNTA_EJEMPLO = "\u00bfQu\u00e9 pide el enunciado del ejercicio sobre b\u00fasqueda binaria?"  # TODO: ajusta a un ejercicio real de tu curso

rag_retrieval_config = genai_types.RagRetrievalConfig(
    top_k=5,
    filter=genai_types.RagRetrievalConfigFilter(vector_distance_threshold=0.5),
)

contexts = client.rag.retrieve_contexts(
    vertex_rag_store=genai_types.VertexRagStore(
        rag_resources=[genai_types.VertexRagStoreRagResource(rag_corpus=rag_corpus.name)],
    ),
    query=types.RagQuery(text=PREGUNTA_EJEMPLO, rag_retrieval_config=rag_retrieval_config),
)
print(contexts)

contexts=RagContexts(
  contexts=[
    RagContextsContext(
      chunk=RagChunk(
        chunk_id='5788074357102564100',
        file_id='5788074357099930365',
        text="""Semana 7 - Arreglos
25 al 28 de agosto 2026
Ejercicios de coding bat: 
Realizar al menos 5 ejercicios de Array-1 de https://codingbat.com/java/Array-1
Ejercicio 1: arreglo básico
Leer el tamaño del arreglo por el usuario (variable x)
Defina un nuevo arreglo de enteros llamado arr_int, de tamaño x.
Pídale al usuario tantos números enteros como el tamaño del arreglo arr_int creado, y asígnele el doble de cada uno de esos datos recibidos a las posiciones del arreglo arr_int.
Realice un ciclo para imprimir todos las variables del arreglo multiplicados por tres por pantalla.
Ejercicio 2: Crear un arreglo invertido a partir de otro arreglo
Cree un arreglo de n enteros (arreglo1) con datos generados a partir de cualquier serie de su preferencia (aproveche la variable del índice i). A partir de este arreglo1, crea otro a

## Paso 6 — Generación anclada (RAG) con Gemini

Ahora sí: le damos el corpus como *herramienta de recuperación* a Gemini para que redacte
una respuesta completa, citando el contenido real de tus enunciados en vez de inventar o
recordar de su entrenamiento general.

In [16]:
GENAI_LOCATION = "global" 

In [17]:
GENAI_LOCATION = "global" 
rag_retrieval_tool = genai_types.Tool(
    retrieval=genai_types.Retrieval(
        vertex_rag_store=genai_types.VertexRagStore(
            rag_resources=[genai_types.VertexRagStoreRagResource(rag_corpus=rag_corpus.name)],
            rag_retrieval_config=rag_retrieval_config,
        ),
    )
)

genai_client = genai.Client(enterprise=True, project=PROJECT_ID, location=GENAI_LOCATION)

response_rag = genai_client.models.generate_content(
    model=MODEL_ID,
    contents=PREGUNTA_EJEMPLO,
    config=genai_types.GenerateContentConfig(tools=[rag_retrieval_tool]),
)
print(response_rag.text)

En los documentos proporcionados no hay un ejercicio titulado explícitamente como "búsqueda binaria". Sin embargo, el **Ejercicio 6 de la Clase 4 (Ciclos Parte 1)** plantea el clásico juego de adivinación que utiliza exactamente la estrategia y lógica de la búsqueda binaria. 

El enunciado de dicho ejercicio pide lo siguiente:

* **Objetivo:** Crear un algoritmo y un programa (en Java) en el que el usuario deba ingresar números enteros de manera sucesiva hasta adivinar un número aleatorio entre 0 y 100 generado al azar por la computadora.
* **Indicaciones del flujo:** El programa debe avisar en cada intento si el número introducido por el usuario es **más grande** o **más pequeño** que el número generado aleatoriamente (lo que permite al usuario ir reduciendo el rango de búsqueda a la mitad, emulando una búsqueda binaria).
* **Resultado final:** Al acertar el número, el programa debe imprimir en pantalla el **número total de intentos** que realizó el usuario para adivinar el número.

A

## Paso 7 (opcional pero recomendado) — Comparar contra el modelo SIN RAG

Esta celda pide la misma pregunta a Gemini **sin** la herramienta de recuperación. La
diferencia entre las dos respuestas es justamente la evidencia de que el conocimiento
específico (el enunciado real de tu curso) viene del corpus y no del modelo base — el
mismo punto que motivó elegir esta tarea para el taller 3.

In [18]:
response_sin_rag = genai_client.models.generate_content(
    model=MODEL_ID,
    contents=PREGUNTA_EJEMPLO,
)

print("--- CON RAG (ancla en tus enunciados) ---")
print(response_rag.text)
print()
print("--- SIN RAG (solo conocimiento del modelo base) ---")
print(response_sin_rag.text)

--- CON RAG (ancla en tus enunciados) ---
En los documentos proporcionados no hay un ejercicio titulado explícitamente como "búsqueda binaria". Sin embargo, el **Ejercicio 6 de la Clase 4 (Ciclos Parte 1)** plantea el clásico juego de adivinación que utiliza exactamente la estrategia y lógica de la búsqueda binaria. 

El enunciado de dicho ejercicio pide lo siguiente:

* **Objetivo:** Crear un algoritmo y un programa (en Java) en el que el usuario deba ingresar números enteros de manera sucesiva hasta adivinar un número aleatorio entre 0 y 100 generado al azar por la computadora.
* **Indicaciones del flujo:** El programa debe avisar en cada intento si el número introducido por el usuario es **más grande** o **más pequeño** que el número generado aleatoriamente (lo que permite al usuario ir reduciendo el rango de búsqueda a la mitad, emulando una búsqueda binaria).
* **Resultado final:** Al acertar el número, el programa debe imprimir en pantalla el **número total de intentos** que real

## Notas y solución de problemas

- **`PERMISSION_DENIED` o `API not enabled`**: revisa que `aiplatform.googleapis.com` esté
  habilitada y que tu cuenta tenga el rol `roles/aiplatform.user` en el proyecto.
- **La importación (Paso 4) parece no traer nada en el Paso 5**: espera unos minutos más;
  es asíncrona. Verifica el estado de los archivos en la consola antes de sospechar del
  código.
- **`400 INVALID_ARGUMENT ... restricted to only allowlisted projects` al crear el
  corpus**: el modo "Spanner" de RAG Engine está limitado a proyectos en lista blanca en
  `us-central1`, `us-east1` y `us-east4` por capacidad. Ya evitamos esto usando
  `us-west1` como región por defecto; si aun así te sale este error (o quieres quedarte en
  una de esas tres regiones), la otra alternativa que ofrece el mensaje de error es pasar
  el corpus a "Serverless mode" — ver
  https://docs.cloud.google.com/gemini-enterprise-agent-platform/build/rag-engine/switching-modes
- **Error de región / recurso no disponible (distinto al de arriba)**: RAG Engine no está
  desplegado en todas las regiones. Consulta la lista de regiones soportadas en
  https://docs.cloud.google.com/vertex-ai/generative-ai/docs/rag-engine/rag-overview y
  ajusta `LOCATION` (y el bucket) de forma consistente.
- **`MODEL_ID` no disponible**: los nombres de modelos Gemini cambian con cada generación;
  si `gemini-3.5-flash` no existe en tu proyecto, revisa el listado de modelos habilitados
  en *Agent Platform → Model Garden* y sustitúyelo.
- **Costos**: tanto el almacenamiento vectorial del corpus como cada llamada de generación
  tienen costo. Para pruebas, usa pocos documentos y limita las preguntas de prueba.